# Getting a dataset into your GCS workspace

You have a private GCS bucket mounted at `~/gcs`. Nobody else can read it. It
outlives your notebook pod, your PVC, and the cluster.

| Path | What | Use it for |
|---|---|---|
| `~` | 20Gi disk | notebooks, code, git checkouts, in-progress checkpoints |
| `~/gcs` | your GCS bucket | datasets, finished artifacts, anything large |
| `~/shared` | shared bucket, read-only | common datasets someone already downloaded |

**Put big things in `~/gcs`, not in `~`.** The home disk is 20Gi and filling it
wedges your notebook.

## 1. Check what you have

In [ ]:
!ls -la ~/gcs
# Your bucket name, straight from the mount:
!mount | grep gcsfuse

In [ ]:
# Your notebook has its own cloud identity -- no keys, no `gcloud auth login`.
# This prints hdlab-elideng.svc.id.goog, NOT your @google.com address. That is correct.
!gcloud auth list

# How to learn your own bucket name. NOT `gcloud storage ls` with no argument:
# that lists the *project's* buckets, needs storage.buckets.list, which you
# deliberately do not have, and returns 403. findmnt reads the name off your
# own mount -- no permission needed, and correct even for a sanitized username.
!findmnt -n -o SOURCE ~/gcs

## 2. Download a dataset

Three ways, in rough order of how much data they suit.

### Small file: curl straight into the mount

Check the `ls` output. `curl -fsSL` is silent by design: if the URL is wrong it
writes nothing, prints nothing, and the next cell fails somewhere else.

In [ ]:
!mkdir -p ~/gcs/datasets
!curl -fsSL https://storage.googleapis.com/cloud-samples-data/ai-platform/iris/iris_data.csv \
    -o ~/gcs/datasets/iris.csv
!ls -lh ~/gcs/datasets/

### Hugging Face: download to the mount

`--local-dir` inside `~/gcs` means the files land in your bucket directly. Set
`HF_HOME` to a path in `~/gcs` too, or the HF cache eats your 20Gi home disk.

The command is `hf`, **not** `huggingface-cli`: this image ships
`huggingface_hub` 1.x, where the old name is a stub that downloads nothing.

In [ ]:
import os
os.environ["HF_HOME"] = os.path.expanduser("~/gcs/.cache/huggingface")

!hf download --repo-type dataset stanfordnlp/imdb \
    --local-dir ~/gcs/datasets/imdb
!du -sh ~/gcs/datasets/imdb

### Large / many files: don't use the mount

gcsfuse is a compatibility shim, not a transfer tool. Above a few GB, or above a
few thousand files, copy bucket-to-bucket with `gcloud storage`, which is
parallel and never touches this pod's disk. This is the single biggest
performance lever in this notebook.

In [ ]:
# Server-side, parallel, no local staging. Replace the source with your own.
import subprocess

MY_BUCKET = subprocess.check_output(
    ["findmnt", "-n", "-o", "SOURCE", "/home/jovyan/gcs"], text=True
).strip()
print("my bucket:", MY_BUCKET)

# Verify it -- this one works, because it names the bucket explicitly:
!gcloud storage ls "gs://{MY_BUCKET}/"

# !gcloud storage rsync -r gs://some-public-dataset "gs://{MY_BUCKET}/datasets/foo"

## 3. Read it back

Two equivalent routes. Use whichever the library in front of you prefers.

In [ ]:
# (a) As ordinary files, through the mount. Anything that takes a path works.
import pandas as pd

df = pd.read_csv("~/gcs/datasets/iris.csv", nrows=5)
df.head()

In [ ]:
# (b) As gs:// URLs via gcsfs -- no mount involved, and it is faster for a
# single large read because it does not go through FUSE.
#
# Reuse MY_BUCKET from the cell above. Discovering the bucket with a bare
# `gcloud storage ls` needs storage.buckets.list, which you deliberately do
# not have -- it raises CalledProcessError on a 403.
df = pd.read_csv(f"gs://{MY_BUCKET}/datasets/iris.csv", nrows=5)
df.head()

## 4. Checkpoints during training

Write checkpoints to the **local disk** and sync the ones you want to keep. A
training loop that writes every step straight into `~/gcs` will be slow and can
stall on a network hiccup -- each small write becomes an object operation.

In [ ]:
# Pattern: fast local scratch, deliberate sync. Needs MY_BUCKET from the cell above.
!mkdir -p ~/scratch/ckpt

# Stand-in for the file a training loop would write. Without it there is nothing
# to sync, and the rsync below prints "Completed files 0 | 0B" -- which tells you
# nothing about whether the pattern works.
!dd if=/dev/urandom of=~/scratch/ckpt/step-1000.pt bs=1M count=8 status=none

!gcloud storage rsync -r ~/scratch/ckpt "gs://{MY_BUCKET}/checkpoints/myrun"
!gcloud storage ls -l "gs://{MY_BUCKET}/checkpoints/myrun/"

## 5. Things that will surprise you

- **`~/gcs` is not POSIX.** No hard links, no atomic rename of a directory, and
  in-place partial writes rewrite the whole object. Don't put a git repo, a
  SQLite database, or a conda env there.
- **Empty directories don't exist.** GCS has no directories; `mkdir ~/gcs/foo`
  can vanish until something is written inside it.
- **`ls` on a huge prefix is slow** — it is a paginated API call, not a directory
  read.
- **You cannot see anyone else's workspace, and that is deliberate.** If you need
  to share a dataset, ask an admin to put it in `~/shared` so it is downloaded
  once for everyone.
- **No quota.** Nothing stops you filling the bucket and running up the bill.
  Delete what you are done with: `gcloud storage rm -r gs://<bucket>/old-thing`.

## 6. Confirm your workspace really is private

Worth running once, so you trust it. Ask a colleague for their bucket name; this
must fail.

In [ ]:
!gcloud storage ls gs://hdlab-elideng-jupyter-someone-else/ \
    && echo "UNEXPECTED: report this to the lab admin" \
    || echo "correctly denied -- your workspace is isolated the same way"